<a href="https://colab.research.google.com/github/sinskid/deep_learning_project/blob/main/notebooks/models_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check if we're running in Colab and set up the environment accordingly
import os
IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !git clone https://github.com/sinskid/deep_learning_project.git
    %cd deep_learning_project
    !pip install -r requirements.txt

import sys
repo_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_path not in sys.path:
    sys.path.append(repo_path)
from src.data import load_testdata, load_traindata
from src.models import get_cnn , get_vit
from src.utils import setup_dirs
from src.train import train
import torch
from torch import nn
from torch import optim

Mounted at /content/drive
Cloning into 'deep_learning_project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 38 (delta 12), reused 32 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 8.77 KiB | 8.77 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/deep_learning_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.2 MB/s eta 0:00:00


**SET-UP FICHIERS**

In [2]:
# Set up directories for outputs, models, and logs
paths = setup_dirs()

**SET-UP MODELS AND DEVICE**

In [4]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
# Initialize the CNN model and move it to the appropriate device
model_cnn = get_cnn().to(device)

# Initialize the ViT model and move it to the appropriate device
model_vit = get_vit().to(device)

cuda


In [5]:
# Dataloader train
dataloader_train = load_traindata(batch_size=32, num_workers=2)

# Dataloader test (no effects on images)
dataloader_test = load_testdata(batch_size=32, num_workers=2)

# Dataloader test (with effects on images)
dataloader_test_blur = load_testdata(batch_size=32, num_workers=2, effects={'blur': 0.5, 'mask': 0.16, 'color': 0.5})

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5400 [00:00<?, ? examples/s]

**ENTRAINEMENT CNN**

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.001)

# NB_EPOCHS Set to 10 for Colab, 1 for local testing
if device == "cuda":
    NB_EPOCHS = 10
else:
    NB_EPOCHS = 1

# Check if a saved model already exists
if os.path.exists(os.path.join(paths['models'], 'cnn_model.pth')):
    print("Cnn Model exists")

# Train the model with early stopping if no model is found in the models directory
else:
    valid_loss = float('inf')
    for epoch in range(NB_EPOCHS):
        train_loss, test_loss = train(model_cnn, dataloader_train, dataloader_test, optimizer, criterion, device)
        print(f"Epoch {epoch+1}/{NB_EPOCHS}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
        # Early stopping condition
        if test_loss < valid_loss:
            valid_loss = test_loss
            torch.save(model_cnn.state_dict(), os.path.join(paths['models'], 'cnn_model.pth'))
        else:
            print("Early stopping triggered.")
            break

Epoch 1/10, Train Loss: 0.4858, Test Loss: 0.3232
